# Algoritmo de Grover con Qiskit e IBM Quantum
por joseluisrg (aprovechando IA)

Para ejecutar este Notebook necesitaras una cuenta gratuita en [IBM Quantum](https://quantum.ibm.com)

En este notebook aprenderás el **Algoritmo de Grover** y por qué es importante contra métodos clásicos. Mencionaremos cómo funciona la **aceleración cuántica** y cómo construir un **oráculo dinámico** que marque uno o varios estados. Usaremos **hardware cuántico real de IBM** y cómo interpretar y **visualizar los resultados**

---

## Conceptos básicos

Antes de comenzar, repasemos tres ideas clave:

| Concepto | Descripción sencilla |
|---|---|
| **Qubit** | La versión cuántica del bit. Puede ser 0, 1, o una **superposición** de ambos al mismo tiempo. |
| **Superposición** | Un qubit puede "ser" 0 y 1 simultáneamente hasta que lo medimos. |
| **Interferencia** | Las probabilidades cuánticas pueden sumarse (constructiva) o cancelarse (destructiva), como olas en el agua. |
| **Oráculo** | Una caja negra cuántica que "reconoce" la solución que buscamos sin revelarla directamente. |

> **Analogía del algoritmo de Grover:** Imagina que tienes 1000 cajas y en una de ellas hay un premio. Clásicamente debes abrir caja por caja (promedio: 500 intentos). Grover te permite encontrarla abriendo apenas ~32 cajas. ¡Eso es la aceleración cuántica!

---

## Paso 1: Instalación de dependencias

Primero instalamos las librerías necesarias. Si ya las tienes instaladas, puedes omitir esta celda.

In [ ]:
# Instalamos Qiskit y los paquetes necesarios para IBM Quantum
# qiskit-ibm-runtime: permite conectarse a hardware real de IBM
# matplotlib: para graficar resultados
# pylatexenc: para renderizar circuitos en formato LaTeX

%pip install qiskit qiskit-ibm-runtime matplotlib pylatexenc --quiet

---

## Paso 2: Dependencias

Importamos todo lo que necesitaremos a lo largo del notebook.

In [ ]:
# ─────────────────────────────────────────────
# Librerías estándar de Python
# ─────────────────────────────────────────────
import math
import time
import random

# ─────────────────────────────────────────────
# Qiskit: el SDK de computación cuántica de IBM
# ─────────────────────────────────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import MCMTGate, ZGate, HGate

# ─────────────────────────────────────────────
# Qiskit IBM Runtime: conexión a hardware real
# ─────────────────────────────────────────────
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2   # simulador local fiel al ruido real

# ─────────────────────────────────────────────
# Visualización
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("✅ Importaciones exitosas")

---

## Paso 3: Configuración — Define tu búsqueda

Aquí es donde defines **qué cadenas de bits quieres encontrar**. El circuito se adaptará automáticamente al número de bits.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN PRINCIPAL — Modifica estos valores a tu gusto
# ─────────────────────────────────────────────────────────────────────────────

# Lista de cadenas de bits que quieres encontrar (los "estados marcados").
# Todas deben tener la misma longitud.
# Ejemplos: ["1101"], ["0011", "1100"], ["101", "010"]
CADENAS_OBJETIVO = ["1101", "0011"]

# ─────────────────────────────────────────────────────────────────────────────
# Validación y cálculo automático de parámetros del circuito
# ─────────────────────────────────────────────────────────────────────────────

# Verificamos que todas las cadenas tengan la misma longitud
longitudes = set(len(s) for s in CADENAS_OBJETIVO)
assert len(longitudes) == 1, "❌ Todas las cadenas objetivo deben tener la misma longitud."
assert all(c in '01' for s in CADENAS_OBJETIVO for c in s), "❌ Solo se permiten caracteres '0' y '1'."

# Número de qubits = longitud de las cadenas
N_QUBITS = len(CADENAS_OBJETIVO[0])

# Espacio de búsqueda: cuántos estados posibles hay con N_QUBITS bits
N_ESTADOS = 2 ** N_QUBITS

# Número de estados marcados (soluciones)
M_SOLUCIONES = len(CADENAS_OBJETIVO)

# Número óptimo de iteraciones de Grover:
# La fórmula es: floor( (π/4) * sqrt(N/M) )
# Esta cantidad maximiza la probabilidad de medir la solución correcta
N_ITERACIONES = max(1, round((math.pi / 4) * math.sqrt(N_ESTADOS / M_SOLUCIONES)))

# ─────────────────────────────────────────────────────────────────────────────
print(f"""📊 Parámetros del experimento:
  • Cadenas objetivo    : {CADENAS_OBJETIVO}
  • Número de qubits    : {N_QUBITS}
  • Espacio de búsqueda : {N_ESTADOS} estados posibles (2^{N_QUBITS})
  • Soluciones marcadas : {M_SOLUCIONES}
  • Iteraciones óptimas : {N_ITERACIONES}
""")

---

## Paso 4: Construcción del Oráculo

### ¿Qué es el oráculo?

El oráculo es el componente central del algoritmo de Grover. Su trabajo es **marcar** los estados que son soluciones, **invirtiendo su fase** (multiplicándolos por −1). Esto no cambia las probabilidades de medición directamente, pero prepara el terreno para que la siguiente etapa (el difusor) amplifique esas soluciones.

Matemáticamente: `|x⟩ → -|x⟩` si `x` es solución, y `|x⟩ → |x⟩` si no lo es.

### ¿Cómo marcamos un estado específico?

Para marcar el estado `|1101⟩` usamos:
1. Compuertas **X** en los qubits que deben ser `0` (para convertirlos en `1` temporalmente)
2. Una compuerta **multi-controlled Z** (MCZ) que invierte la fase cuando **todos** los qubits son `1`
3. Compuertas **X** nuevamente para revertir el paso 1

In [ ]:
def construir_oraculo(n_qubits: int, cadenas_objetivo: list[str]) -> QuantumCircuit:
    """
    Construye el oráculo de Grover de forma dinámica.

    El oráculo aplica una inversión de fase (-1) a cada estado que coincide
    con alguna de las cadenas objetivo. Esto se logra mediante:
      1. Compuertas X en las posiciones con '0' (para normalizar a |111...1⟩)
      2. Una compuerta MCZ (Multi-Controlled-Z) que actúa sobre todos los qubits
      3. Las mismas compuertas X para deshacer la normalización

    Parámetros:
    -----------
    n_qubits : int
        Número de qubits del circuito.
    cadenas_objetivo : list[str]
        Lista de cadenas de bits (ej. ["1101", "0011"]) que serán marcadas.
        Cada carácter corresponde a un qubit (índice 0 = qubit más significativo).

    Retorna:
    --------
    QuantumCircuit
        El subcircuito del oráculo, listo para ser añadido al circuito principal.
    """
    # Creamos un circuito vacío para el oráculo
    qc = QuantumCircuit(n_qubits, name="Oráculo")

    for cadena in cadenas_objetivo:
        # Invertimos (bit a bit) el orden de la cadena porque Qiskit indexa
        # el qubit 0 como el BIT MENOS significativo (el de la derecha).
        # Así, la cadena "1101" corresponde a:
        #   qubit 0 → '1' (posición 3), qubit 1 → '0', qubit 2 → '1', qubit 3 → '1'
        cadena_invertida = cadena[::-1]

        # Paso 1: Aplicar X en qubits donde el bit objetivo es '0'
        # Esto convierte ese qubit de 0→1, para que la MCZ pueda actuar
        for i, bit in enumerate(cadena_invertida):
            if bit == '0':
                qc.x(i)

        # Paso 2: Compuerta MCZ (Multi-Controlled-Z)
        # Invierte la fase del estado |111...1⟩
        # Usamos MCMTGate con la compuerta Z sobre 1 qubit objetivo
        # y los demás como controles
        if n_qubits == 1:
            # Caso especial: con 1 solo qubit, una Z simple es suficiente
            qc.z(0)
        else:
            # Qubits de control: todos menos el último
            controles = list(range(n_qubits - 1))
            # Qubit objetivo de la Z: el último qubit
            objetivo = n_qubits - 1
            # MCMTGate: aplica ZGate controlada por 'controles'
            mcz = MCMTGate(ZGate(), num_ctrl_qubits=len(controles), num_target_qubits=1)
            qc.append(mcz, controles + [objetivo])

        # Paso 3: Deshacer las X del Paso 1 (el oráculo debe ser reversible)
        for i, bit in enumerate(cadena_invertida):
            if bit == '0':
                qc.x(i)

        # Separador visual entre marcadores (solo cosmético)
        if len(cadenas_objetivo) > 1:
            qc.barrier()

    return qc


# Construimos el oráculo con los parámetros definidos arriba
oraculo = construir_oraculo(N_QUBITS, CADENAS_OBJETIVO)

print("✅ Oráculo construido exitosamente")
print(f"   Profundidad del oráculo: {oraculo.depth()} capas")
print(f"   Compuertas usadas: {dict(oraculo.count_ops())}")
print()
print("📐 Vista del oráculo:")
oraculo.draw('mpl', style='iqp', fold=40)

---

## Paso 5: Construcción del Difusor

### ¿Qué es el difusor?

El difusor (también llamado **operador de Grover** o **reflexión sobre el estado uniforme**) es la segunda etapa del algoritmo. Su función es **amplificar la amplitud** de los estados marcados por el oráculo y **reducir la de los demás**.

Matemáticamente aplica la transformación: `2|s⟩⟨s| - I`

donde `|s⟩` es la superposición uniforme de todos los estados.

### Implementación paso a paso:

```
H⊗n  →  X⊗n  →  MCZ  →  X⊗n  →  H⊗n
```

In [ ]:
def construir_difusor(n_qubits: int) -> QuantumCircuit:
    """
    Construye el operador de difusión de Grover.

    Este operador realiza una reflexión alrededor del estado de superposición
    uniforme |s⟩ = H⊗n|0...0⟩. En términos matriciales aplica: 2|s⟩⟨s| - I

    La implementación es:
      H⊗n → X⊗n → MCZ → X⊗n → H⊗n

    Parámetros:
    -----------
    n_qubits : int
        Número de qubits del circuito.

    Retorna:
    --------
    QuantumCircuit
        El subcircuito del difusor.
    """
    qc = QuantumCircuit(n_qubits, name="Difusor")

    # Paso 1: Hadamard en todos los qubits
    # Transforma |s⟩ (superposición uniforme) al estado |0...0⟩
    qc.h(range(n_qubits))

    # Paso 2: X en todos los qubits
    # Convierte |0...0⟩ → |1...1⟩ para que la MCZ actúe sobre él
    qc.x(range(n_qubits))

    # Paso 3: MCZ — invierte la fase del estado |1...1⟩ (= |0...0⟩ original)
    if n_qubits == 1:
        qc.z(0)
    else:
        controles = list(range(n_qubits - 1))
        objetivo = n_qubits - 1
        mcz = MCMTGate(ZGate(), num_ctrl_qubits=len(controles), num_target_qubits=1)
        qc.append(mcz, controles + [objetivo])

    # Paso 4: X de vuelta (operación inversa del Paso 2)
    qc.x(range(n_qubits))

    # Paso 5: Hadamard de vuelta (operación inversa del Paso 1)
    qc.h(range(n_qubits))

    return qc


difusor = construir_difusor(N_QUBITS)

print("✅ Difusor construido exitosamente")
print(f"   Profundidad del difusor: {difusor.depth()} capas")
print()
print("📐 Vista del difusor:")
difusor.draw('mpl', style='iqp', fold=40)

---

## Paso 6: Ensamblar el Circuito Completo de Grover

Ahora unimos todas las piezas:

```
  |0⟩^n  →  H^n  →  [Oráculo + Difusor] × k  →  Medición
                      ↑________________________↑
                         k iteraciones óptimas
```

In [ ]:
def construir_circuito_grover(
    n_qubits: int,
    cadenas_objetivo: list[str],
    n_iteraciones: int
) -> QuantumCircuit:
    """
    Ensambla el circuito completo del algoritmo de Grover.

    Estructura del circuito:
      1. Estado inicial: |0⟩^n
      2. Superposición uniforme: H^n
      3. k iteraciones de: Oráculo → Difusor
      4. Medición de todos los qubits

    Parámetros:
    -----------
    n_qubits : int
        Número de qubits.
    cadenas_objetivo : list[str]
        Cadenas de bits que se quieren encontrar.
    n_iteraciones : int
        Número de veces que se aplica el par Oráculo+Difusor.

    Retorna:
    --------
    QuantumCircuit
        El circuito completo de Grover con mediciones.
    """
    # Registros cuánticos y clásicos
    qr = QuantumRegister(n_qubits, name='q')
    cr = ClassicalRegister(n_qubits, name='c')
    qc = QuantumCircuit(qr, cr, name="Grover")

    # ── ETAPA 1: Inicialización ─────────────────────────────────────────────
    # Todos los qubits comienzan en |0⟩.
    # Aplicamos Hadamard a cada qubit para crear la superposición uniforme:
    # |s⟩ = (1/√N) Σ|x⟩  (todos los estados con la misma probabilidad)
    qc.h(qr)
    qc.barrier(label="Inicialización")

    # ── ETAPA 2: Iteraciones de Grover ─────────────────────────────────────
    oraculo = construir_oraculo(n_qubits, cadenas_objetivo)
    difusor  = construir_difusor(n_qubits)

    for i in range(n_iteraciones):
        # Convertimos los subcircuitos en instrucciones (cajas) para
        # que el diagrama sea más legible
        qc.append(oraculo.to_instruction(), qr)
        qc.append(difusor.to_instruction(), qr)
        qc.barrier(label=f"Iter {i+1}")

    # ── ETAPA 3: Medición ───────────────────────────────────────────────────
    # Colapsamos el estado cuántico y leemos los valores clásicos
    qc.measure(qr, cr)

    return qc


# Construimos el circuito con los parámetros configurados
circuito_grover = construir_circuito_grover(N_QUBITS, CADENAS_OBJETIVO, N_ITERACIONES)

print(f"✅ Circuito de Grover ensamblado")
print(f"   Qubits       : {circuito_grover.num_qubits}")
print(f"   Profundidad  : {circuito_grover.depth()} capas")
print(f"   Compuertas   : {dict(circuito_grover.count_ops())}")

---

## Paso 7: Visualización del Circuito

Visualizamos el circuito completo. Los bloques "Oráculo" y "Difusor" se muestran como cajas para mayor claridad.

In [ ]:
print("📐 Circuito de Grover (vista compacta con bloques):")
circuito_grover.draw('mpl', style='iqp', fold=50, scale=0.8)

In [ ]:
# Vista expandida: descompone los bloques en compuertas individuales
# Útil para ver exactamente qué compuertas se ejecutan
print("📐 Circuito de Grover (vista expandida — compuertas individuales):")
circuito_grover.decompose(reps=2).draw('mpl', style='iqp', fold=60, scale=0.7)

---

## Paso 8: Transpilación (Pass Manager)

### ¿Qué es la transpilación?

Los procesadores cuánticos reales tienen limitaciones:
- Solo soportan un **conjunto específico de compuertas** nativas
- Solo ciertos pares de qubits pueden interactuar directamente (**conectividad**)
- Hay niveles de **ruido** que afectan los resultados

El **Pass Manager** (transpilador) convierte nuestro circuito ideal a uno que el hardware puede ejecutar, optimizando al mismo tiempo su profundidad y número de compuertas.

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Usamos FakeManilaV2: un simulador que imita fielmente el ruido
# y la topología del chip cuántico Manila de IBM (5 qubits).
# Es ideal para probar con pocos qubits sin consumir créditos reales.
backend_simulado = FakeManilaV2()

# El Pass Manager de optimización nivel 3 intenta minimizar
# la profundidad del circuito agresivamente. Niveles: 0 (mínimo) → 3 (máximo)
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend_simulado
)

# Transpilamos el circuito al hardware objetivo
circuito_transpilado = pass_manager.run(circuito_grover)

# Comparamos métricas antes y después de la transpilación
print("📊 Comparativa antes/después de transpilación:")
print(f"{'Métrica':<30} {'Original':>12} {'Transpilado':>12}")
print("-" * 56)
print(f"{'Profundidad (capas)':<30} {circuito_grover.depth():>12} {circuito_transpilado.depth():>12}")
print(f"{'Núm. compuertas totales':<30} {sum(circuito_grover.count_ops().values()):>12} {sum(circuito_transpilado.count_ops().values()):>12}")
print(f"{'Núm. qubits':<30} {circuito_grover.num_qubits:>12} {circuito_transpilado.num_qubits:>12}")
print()
print(f"Compuertas nativas usadas: {dict(circuito_transpilado.count_ops())}")

In [ ]:
print("📐 Circuito transpilado (compuertas nativas del hardware):")
circuito_transpilado.draw('mpl', style='iqp', fold=80, scale=0.5)

---

## Paso 9: Ejecución en Hardware

### Opción A — Simulador con ruido (local, sin cuenta IBM)
Ideal para pruebas rápidas.

### Opción B — Hardware cuántico real de IBM
Requiere cuenta en [IBM Quantum](https://quantum.ibm.com) y tu API token.

> ⚡ **Recomendación para principiantes:** Comienza con la Opción A para verificar que el circuito funciona, y luego prueba con hardware real en la Opción B.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🔧 ELIGE EL MODO DE EJECUCIÓN
# ─────────────────────────────────────────────────────────────────────────────
USAR_HARDWARE_REAL = False   # Cambia a True para usar IBM Quantum real
IBM_API_TOKEN = ""           # Pega aquí tu API token de https://quantum.ibm.com
N_SHOTS = 1024

resultados_cuentas = {}

if not USAR_HARDWARE_REAL:
    # ── OPCIÓN A: Simulador local con modelo de ruido ──────────────────────
    print("🖥️  Ejecutando en simulador FakeManilaV2 (con ruido realista)...")

    # ✅ FIX: usar mode= en lugar de backend=
    sampler = Sampler(mode=backend_simulado)
    sampler.options.default_shots = N_SHOTS

    job = sampler.run([circuito_transpilado])
    resultado = job.result()

    pub_result = resultado[0]
    resultados_cuentas = pub_result.data.c.get_counts()

    print(f"✅ Simulación completada")
    print(f"   Top 5 estados: {dict(sorted(resultados_cuentas.items(), key=lambda x: -x[1])[:5])}")

else:
    # ── OPCIÓN B: Hardware cuántico real de IBM ────────────────────────────
    assert IBM_API_TOKEN, "❌ Debes proporcionar tu IBM_API_TOKEN."

    print("🌐 Conectando a IBM Quantum...")
    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token=IBM_API_TOKEN
    )

    backend_real = service.least_busy(
        operational=True,
        simulator=False,
        min_num_qubits=N_QUBITS
    )
    print(f"✅ Backend seleccionado: {backend_real.name}")

    # Retranspilamos al backend real
    pm_real = generate_preset_pass_manager(
        optimization_level=3,
        backend=backend_real
    )
    circuito_real = pm_real.run(circuito_grover)

    print(f"\n⏳ Enviando trabajo ({N_SHOTS} shots)...")

    # ✅ FIX: usar mode= en lugar de backend=
    sampler_real = Sampler(mode=backend_real)
    sampler_real.options.default_shots = N_SHOTS

    job_real = sampler_real.run([circuito_real])
    print(f"   Job ID: {job_real.job_id()}")
    print("   Esperando resultados...")

    resultado_real = job_real.result()
    resultados_cuentas = resultado_real[0].data.c.get_counts()

    print(f"✅ Resultados recibidos")
    print(f"   Top 5 estados: {dict(sorted(resultados_cuentas.items(), key=lambda x: -x[1])[:5])}")

---

## Paso 10: Visualización de Resultados

Graficamos el histograma de mediciones. Los estados objetivo deberían aparecer con una frecuencia significativamente mayor que el resto.

In [ ]:
def graficar_histograma(cuentas: dict, cadenas_objetivo: list[str], n_shots: int):
    """
    Genera un histograma con los resultados del experimento cuántico.

    Los estados marcados como objetivo se colorean en naranja para
    distinguirlos visualmente del resto.

    Parámetros:
    -----------
    cuentas : dict
        Diccionario {cadena_bits: número_de_ocurrencias}.
    cadenas_objetivo : list[str]
        Los estados que el algoritmo debía encontrar.
    n_shots : int
        Número total de mediciones realizadas.
    """
    # Ordenamos los estados para que el eje x sea consistente
    estados = sorted(cuentas.keys())
    frecuencias = [cuentas.get(e, 0) for e in estados]
    probabilidades = [f / n_shots for f in frecuencias]

    # Colores: naranja para objetivos, azul para el resto
    colores = ['#FF6B35' if e in cadenas_objetivo else '#4A90D9' for e in estados]

    fig, ax = plt.subplots(figsize=(max(10, len(estados) * 0.6), 6))

    barras = ax.bar(estados, probabilidades, color=colores,
                    edgecolor='white', linewidth=0.8, alpha=0.92)

    # Etiquetas de probabilidad encima de cada barra
    for barra, prob in zip(barras, probabilidades):
        if prob > 0.005:  # Solo mostrar si es visible
            ax.text(
                barra.get_x() + barra.get_width() / 2,
                barra.get_height() + 0.005,
                f'{prob:.2%}',
                ha='center', va='bottom', fontsize=9, fontweight='bold'
            )

    # Línea de probabilidad uniforme (lo que esperaríamos sin Grover)
    prob_uniforme = 1 / (2 ** len(estados[0]))
    ax.axhline(y=prob_uniforme, color='gray', linestyle='--', linewidth=1.2,
               label=f'Prob. uniforme (sin Grover): {prob_uniforme:.2%}')

    # Leyenda
    parche_obj = mpatches.Patch(color='#FF6B35', label=f'Estados objetivo: {cadenas_objetivo}')
    parche_otro = mpatches.Patch(color='#4A90D9', label='Otros estados')
    ax.legend(handles=[parche_obj, parche_otro,
                        plt.Line2D([0], [0], color='gray', linestyle='--',
                                   label=f'Prob. uniforme: {prob_uniforme:.2%}')],
              loc='upper right', fontsize=10)

    # Probabilidades teóricas de los objetivos (ideal sin ruido)
    prob_teorica = sum(probabilidades[estados.index(e)]
                       for e in cadenas_objetivo if e in estados)

    ax.set_title(
        f'Resultados del Algoritmo de Grover\n'
        f'Objetivos: {cadenas_objetivo} | '
        f'Prob. medida objetivos: {prob_teorica:.2%} | '
        f'Shots: {n_shots}',
        fontsize=13, fontweight='bold', pad=14
    )
    ax.set_xlabel('Estado medido (cadena de bits)', fontsize=12)
    ax.set_ylabel('Probabilidad de medición', fontsize=12)
    ax.set_ylim(0, min(1.0, max(probabilidades) * 1.25))
    ax.tick_params(axis='x', rotation=45, labelsize=9)

    plt.tight_layout()
    plt.savefig('resultados_grover.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 Gráfico guardado como 'resultados_grover.png'")


graficar_histograma(resultados_cuentas, CADENAS_OBJETIVO, N_SHOTS)

---

## Paso 11: Comparación de Complejidad — Grover vs. Búsqueda Clásica

Una de las mayores ventajas del algoritmo de Grover es su **aceleración cuadrática** frente a la búsqueda clásica no estructurada.

| Algoritmo | Complejidad | Ejemplo: N=1,000,000 estados |
|---|---|---|
| **Búsqueda clásica** | O(N) | ~500,000 consultas en promedio |
| **Algoritmo de Grover** | O(√N) | ~785 iteraciones |

Esto no es una aceleración exponencial, pero sí es **cuadráticamente más rápido**, lo que se vuelve significativo para N muy grandes.

In [ ]:
def busqueda_clasica_simulada(n_qubits: int, objetivo: str, n_simulaciones: int = 1000) -> float:
    """
    Simula una búsqueda clásica aleatoria para comparar con Grover.

    Genera números aleatorios en el espacio de búsqueda hasta encontrar
    el objetivo, y devuelve el promedio de intentos sobre múltiples simulaciones.

    Parámetros:
    -----------
    n_qubits : int
        Número de bits/qubits (define el espacio de búsqueda 2^n).
    objetivo : str
        La cadena de bits objetivo (ej. "1101").
    n_simulaciones : int
        Cuántas veces repetir el experimento para promediar.

    Retorna:
    --------
    float
        Promedio de intentos necesarios para encontrar el objetivo.
    """
    n_estados = 2 ** n_qubits
    objetivo_int = int(objetivo, 2)
    intentos_totales = 0

    for _ in range(n_simulaciones):
        intentos = 0
        while True:
            intentos += 1
            if random.randint(0, n_estados - 1) == objetivo_int:
                break
        intentos_totales += intentos

    return intentos_totales / n_simulaciones


# ─────────────────────────────────────────────────────────────────────────────
# Generamos la comparativa para distintos tamaños de búsqueda
# ─────────────────────────────────────────────────────────────────────────────
n_bits_rango = list(range(2, 20))   # De 2 bits (4 estados) a 19 bits (524,288 estados)

consultas_clasicas = [2**n / 2 for n in n_bits_rango]          # O(N/2) en promedio
consultas_grover   = [math.pi/4 * math.sqrt(2**n) for n in n_bits_rango]  # O(√N * π/4)

# Calculamos el factor de aceleración
aceleracion = [c/g for c, g in zip(consultas_clasicas, consultas_grover)]

# ─────────────────────────────────────────────────────────────────────────────
# Gráfica comparativa
# ─────────────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Número de consultas en escala logarítmica
ax1.semilogy(n_bits_rango, consultas_clasicas, 'o-', color='#E74C3C',
             linewidth=2.5, markersize=6, label='Búsqueda clásica O(N)')
ax1.semilogy(n_bits_rango, consultas_grover, 's-', color='#2ECC71',
             linewidth=2.5, markersize=6, label='Algoritmo de Grover O(√N)')

# Marcamos nuestro experimento actual
ax1.axvline(x=N_QUBITS, color='purple', linestyle=':', linewidth=1.8,
            label=f'Nuestro experimento ({N_QUBITS} qubits)')

ax1.set_xlabel('Número de qubits / bits', fontsize=12)
ax1.set_ylabel('Consultas necesarias (escala log)', fontsize=12)
ax1.set_title('Consultas necesarias:\nBúsqueda clásica vs. Grover', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Gráfica 2: Factor de aceleración
ax2.plot(n_bits_rango, aceleracion, 'D-', color='#9B59B6',
         linewidth=2.5, markersize=6)
ax2.fill_between(n_bits_rango, aceleracion, alpha=0.15, color='#9B59B6')
ax2.axvline(x=N_QUBITS, color='purple', linestyle=':', linewidth=1.8,
            label=f'Nuestro exp. ({N_QUBITS} qubits): {aceleracion[N_QUBITS-2]:.1f}x')

ax2.set_xlabel('Número de qubits / bits', fontsize=12)
ax2.set_ylabel('Factor de aceleración cuántica', fontsize=12)
ax2.set_title('Factor de Aceleración:\n(Clásico / Grover)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Complejidad: Búsqueda Clásica O(N) vs. Algoritmo de Grover O(√N)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparativa_complejidad.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen
print("\n📊 Tabla comparativa para tamaños de búsqueda comunes:")
print(f"{'N (qubits)':<12} {'Estados (N)':<14} {'Clásico O(N/2)':<18} {'Grover O(√N·π/4)':<20} {'Aceleración':<12}")
print("-" * 78)
for n in [4, 8, 16, 20, 32, 64, 128, 256]:
    n_est = 2**n
    cl = n_est / 2
    gr = math.pi/4 * math.sqrt(n_est)
    acc = cl / gr
    print(f"{n:<12} {n_est:<14,.0f} {cl:<18,.0f} {gr:<20,.0f} {acc:<12.1f}x")

---

## Paso 12: Evolución de Probabilidades por Iteración

Una visualización clave: ¿cómo crecen las probabilidades de los estados objetivo conforme aumentan las iteraciones de Grover?

In [ ]:
def prob_teorica_grover(n_qubits: int, m_soluciones: int, k_iter: int) -> float:
    """
    Calcula la probabilidad teórica de medir una solución después de k iteraciones.

    Fórmula: sin²((2k+1) · θ)  donde θ = arcsin(√(M/N))

    Parámetros:
    -----------
    n_qubits : int
        Número de qubits.
    m_soluciones : int
        Número de estados marcados.
    k_iter : int
        Número de iteraciones de Grover.

    Retorna:
    --------
    float
        Probabilidad de medir un estado solución.
    """
    N = 2 ** n_qubits
    theta = math.asin(math.sqrt(m_soluciones / N))
    return math.sin((2 * k_iter + 1) * theta) ** 2


# Calculamos la probabilidad teórica para 0 hasta 3*N_ITERACIONES iteraciones
max_iter = max(20, N_ITERACIONES * 3)
iteraciones = list(range(0, max_iter + 1))
probabilidades_teo = [prob_teorica_grover(N_QUBITS, M_SOLUCIONES, k) for k in iteraciones]

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(iteraciones, probabilidades_teo, '-', color='#2ECC71',
        linewidth=2.5, label='Probabilidad teórica de éxito')
ax.fill_between(iteraciones, probabilidades_teo, alpha=0.12, color='#2ECC71')

# Marcamos el número óptimo de iteraciones
prob_optima = prob_teorica_grover(N_QUBITS, M_SOLUCIONES, N_ITERACIONES)
ax.axvline(x=N_ITERACIONES, color='#E74C3C', linestyle='--', linewidth=2,
           label=f'Iteración óptima (k={N_ITERACIONES}) → P={prob_optima:.2%}')
ax.scatter([N_ITERACIONES], [prob_optima], color='#E74C3C', s=100, zorder=5)

# Línea de probabilidad uniforme (inicio)
ax.axhline(y=M_SOLUCIONES/2**N_QUBITS, color='gray', linestyle=':',
           linewidth=1.5, label=f'Inicio (superposición uniforme): {M_SOLUCIONES/2**N_QUBITS:.2%}')

ax.set_xlabel('Número de iteraciones de Grover', fontsize=12)
ax.set_ylabel('Probabilidad de medir el estado objetivo', fontsize=12)
ax.set_title(
    f'Evolución de la Probabilidad de Éxito\n'
    f'({N_QUBITS} qubits, {M_SOLUCIONES} solución(es) de {2**N_QUBITS} posibles)',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('evolucion_probabilidad.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🎯 Con {N_ITERACIONES} iteración(es), la probabilidad teórica de encontrar")
print(f"   alguno de los {M_SOLUCIONES} estado(s) objetivo es: {prob_optima:.2%}")
print()
print("💡 Nota: Si se hacen demasiadas iteraciones, la probabilidad BAJA nuevamente.")
print("   El algoritmo oscila — por eso el número óptimo de iteraciones es tan importante.")

---

## Paso 13: Resumen y Conclusiones

### Lo que aprendiste en este notebook:

| Componente | Función | Implementación |
|---|---|---|
| **Inicialización** | Crear superposición uniforme | `H⊗n` sobre `\|0⟩^n` |
| **Oráculo** | Marcar estados solución invirtiendo su fase | `X` + `MCZ` + `X` |
| **Difusor** | Amplificar estados marcados | `H⊗n → X⊗n → MCZ → X⊗n → H⊗n` |
| **Iteraciones** | Repetir oráculo+difusor | k = `⌊(π/4)√(N/M)⌋` veces |
| **Medición** | Colapsar el estado cuántico | `measure` |

### Puntos clave:

1. **Aceleración cuadrática**: Grover reduce O(N) → O(√N). No es exponencial, pero sí muy significativa.
2. **El oráculo es dinámico**: Puede marcar cualquier combinación de estados sin conocer el espacio de búsqueda.
3. **Las iteraciones importan**: Demasiadas o muy pocas reduce la probabilidad de éxito.
4. **El ruido del hardware real afecta los resultados**: Por eso la transpilación optimiza el circuito.

### ¿Cuándo usar Grover?

- Búsqueda en bases de datos no estructuradas
- Inversión de funciones criptográficas
- Optimización combinatoria (como subrutina)

---

## 🚀 Próximos pasos

Si quieres profundizar, te recomendamos explorar:

- 📖 [Qiskit Textbook — Grover's Algorithm](https://learning.quantum.ibm.com/course/fundamentals-of-quantum-algorithms/grovers-algorithm)
- 🔬 [IBM Quantum Learning](https://learning.quantum.ibm.com)
- 🎓 [Algoritmo de Shor](https://en.wikipedia.org/wiki/Shor%27s_algorithm) (factorización con aceleración exponencial)
- ⚡ [QAOA](https://en.wikipedia.org/wiki/Quantum_approximate_optimization_algorithm) (optimización cuántica aproximada)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Resumen final del experimento
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("        RESUMEN DEL EXPERIMENTO DE GROVER")
print("=" * 60)
print(f"  Cadenas objetivo    : {CADENAS_OBJETIVO}")
print(f"  Qubits              : {N_QUBITS}")
print(f"  Espacio de búsqueda : {N_ESTADOS} estados")
print(f"  Iteraciones óptimas : {N_ITERACIONES}")
print(f"  Shots realizados    : {N_SHOTS}")
print()

# Probabilidad medida de los objetivos
total_objetivo = sum(resultados_cuentas.get(obj, 0) for obj in CADENAS_OBJETIVO)
prob_medida = total_objetivo / N_SHOTS
prob_teo = prob_teorica_grover(N_QUBITS, M_SOLUCIONES, N_ITERACIONES)

print(f"  Prob. teórica (ideal)    : {prob_teo:.2%}")
print(f"  Prob. medida (experimento): {prob_medida:.2%}")
print(f"  Diferencia (ruido, etc.) : {abs(prob_teo - prob_medida):.2%}")
print()
print(f"  Complejidad clásica   : O(N)  → {N_ESTADOS//2:,} consultas promedio")
print(f"  Complejidad de Grover : O(√N) → {N_ITERACIONES} consultas")
print(f"  Factor de aceleración : ~{(N_ESTADOS//2)/N_ITERACIONES:.1f}x más rápido")
print("=" * 60)